# MediQ Stage 1 DistilBERT Symptom Model Training Pipeline
## 49-Class DDXPlus Pathology Classification
This notebook trains the Stage 1 DistilBERT model for multi-class symptom-to-condition classification using **DDXPlus** only.
Training checkpoints, runs, evaluation results, and model exports are saved directly to **persistent Google Drive storage**.

**Architecture**:
- Base: distilbert-base-uncased
- Target: DDXPlus PATHOLOGY (49 classes)
- Input: Patient presentation text (age, sex, initial symptom, positive clinical evidences)
- Leakage prevention: DIFFERENTIAL_DIAGNOSIS and PATHOLOGY are strictly excluded from input text features.
- Future Stage 2 Compatibility: 49 classifier weights will transfer into 71-class unified space with S2D and DDX replay.

> **CRITICAL**: The training cell is commented out. Google Drive persistence is verified before training begins.


## Step 1: Install Required Dependencies
Installs Hugging Face Transformers, Accelerate, Datasets, Scikit-Learn, and Matplotlib.


In [ ]:
!pip install -q \
    transformers==4.36.2 \
    accelerate==0.25.0 \
    datasets==2.16.1 \
    scikit-learn==1.3.2 \
    matplotlib==3.8.2 \
    tqdm

import transformers, accelerate, sklearn, torch
print(f'PyTorch: {torch.__version__} (CUDA: {torch.cuda.is_available()})')
print(f'Transformers: {transformers.__version__}')
print(f'Accelerate: {accelerate.__version__}')
print(f'Scikit-learn: {sklearn.__version__}')


## Step 2: Mount Google Drive
Mount Google Drive to provide persistent storage for checkpoints, runs, and evaluations.


In [ ]:
from google.colab import drive
import os

drive_mount = '/content/drive'
drive.mount(drive_mount)
print('Google Drive mounted successfully.')


## Step 3: Verify Google Drive & Hierarchy Write Test
Verifies /content/drive and /content/drive/MyDrive exist, creates the standard MediQ/training/symptom/stage1 directories, and verifies write access.


In [ ]:
from pathlib import Path

drive_base = Path('/content/drive/MyDrive')
assert drive_base.exists(), f'MyDrive not found at {drive_base}'

stage1_root = drive_base / 'MediQ' / 'training' / 'symptom' / 'stage1'
dirs = {
    'runs': stage1_root / 'runs',
    'checkpoints': stage1_root / 'checkpoints',
    'evaluation': stage1_root / 'evaluation',
    'exports': stage1_root / 'exports',
}

for name, p in dirs.items():
    p.mkdir(parents=True, exist_ok=True)
    test_f = p / '.write_test'
    test_f.write_text('mediq_ok', encoding='utf-8')
    assert test_f.read_text() == 'mediq_ok'
    test_f.unlink()
    print(f'[OK] Verified persistent directory: {p}')

print('\nAll Google Drive persistent directories verified and writable!')


## Step 4: Define Paths
Set local and Drive paths for the MediQ workspace, processed dataset, and outputs.


In [ ]:
# Configure paths
WORKSPACE_DIR = Path('/content/MediQ')  # Cloned repo or uploaded MediQ project
DATA_DIR = Path('/content/drive/MyDrive/MediQ/processed')  # Persistent processed data location
LOCAL_DATA_DIR = Path('/content/processed')               # Fast local scratch SSD (optional)
DRIVE_ROOT = stage1_root

print(f'Workspace Dir : {WORKSPACE_DIR}')
print(f'Persistent Data Dir : {DATA_DIR}')
print(f'Drive Root : {DRIVE_ROOT}')


## Step 5: Verify / Copy Processed Datasets
Ensures the preprocessed DDXPlus, S2D, and Kerala JSONL files are available.


In [ ]:
# If data is on Drive, verify files; if uploading zip, unzip to LOCAL_DATA_DIR
active_data_dir = DATA_DIR if DATA_DIR.exists() else LOCAL_DATA_DIR

required_files = [
    active_data_dir / 'ddxplus' / 'train' / 'ddx_train.jsonl',
    active_data_dir / 'ddxplus' / 'validate' / 'ddx_validate.jsonl',
    active_data_dir / 'ddxplus' / 'test' / 'ddx_test.jsonl',
    active_data_dir / 'label_mapping' / 'stage1_label_mapping.json',
]

for rf in required_files:
    assert rf.exists(), f'Missing required dataset file: {rf}'
    print(f'[OK] Found: {rf} ({rf.stat().st_size:,} bytes)')


## Step 6: Run Preflight Validation (--validate_only)
Runs complete pipeline validation without training:
- Checks 49 classes
- Checks text fields (no DIFFERENTIAL_DIAGNOSIS)
- Loads tokenizer
- Performs in-memory forward pass test
- Checks Drive persistence


In [ ]:
%cd /content/MediQ/backend/training/symptom
!python train.py --validate_only --google_drive --data_dir {active_data_dir} --drive_root {DRIVE_ROOT}


## Step 7: Confirm GPU
Ensures a fast NVIDIA GPU (T4, V100, A100, or L4) is allocated.


In [ ]:
import torch
if torch.cuda.is_available():
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('WARNING: CUDA is not available. Please go to Runtime -> Change runtime type -> T4 GPU.')


## Step 8: Configure Training Hyperparameters
Configure epochs, batch size, learning rate, and sequence length.


In [ ]:
EPOCHS = 3
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
MAX_SEQ_LENGTH = 128
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
GRADIENT_ACCUMULATION_STEPS = 1
SEED = 42

print('Hyperparameters configured.')


## Step 9: Start Training (Save Directly to Google Drive)
Trains distilbert-base-uncased on DDXPlus 49 pathologies and writes checkpoints directly to Drive.

> **Note**: Uncomment the command below when ready to start GPU fine-tuning.


In [ ]:
# Uncomment to run training:
# !python train.py \
#     --google_drive \
#     --drive_root {DRIVE_ROOT} \
#     --data_dir {active_data_dir} \
#     --epochs {EPOCHS} \
#     --batch_size {BATCH_SIZE} \
#     --learning_rate {LEARNING_RATE} \
#     --max_seq_length {MAX_SEQ_LENGTH} \
#     --weight_decay {WEIGHT_DECAY} \
#     --warmup_ratio {WARMUP_RATIO} \
#     --gradient_accumulation_steps {GRADIENT_ACCUMULATION_STEPS} \
#     --seed {SEED}


## Step 10: Evaluate Official Untouched DDXPlus Test Split
Evaluates the fine-tuned model against all 134,529 samples of the official benchmark.


In [ ]:
# Evaluate official test split
# !python evaluate.py --split test --google_drive --drive_root {DRIVE_ROOT} --data_dir {active_data_dir}


## Step 11: Evaluate Strict Generalization Test Split
Evaluates DDXPlus test set with train<->test exact symptom fingerprints excluded in-memory (dynamic count).


In [ ]:
# Evaluate strict generalization split (in-memory filtering)
# !python evaluate.py --split strict_test --google_drive --drive_root {DRIVE_ROOT} --data_dir {active_data_dir}


## Step 12: External Evaluation: Symptom2Disease Test Set
Evaluates external S2D test set (24 classes: Pneumonia and GERD in-label-space; 22 classes out-of-label-space).


In [ ]:
# External S2D evaluation
# !python evaluate.py --split s2d --google_drive --drive_root {DRIVE_ROOT} --data_dir {active_data_dir}


## Step 13: External Evaluation: Kerala Evaluation Dataset
Evaluates external Kerala clinical profiles (3 out-of-label-space conditions: Chikungunya, Leptospirosis, Nipah).


In [ ]:
# External Kerala evaluation
# !python evaluate.py --split kerala --google_drive --drive_root {DRIVE_ROOT} --data_dir {active_data_dir}


## Step 14: Export Model Artifacts
Exports best checkpoint, config, tokenizer, label mapping, and model_metadata.json to persistent exports and backend models directory.


In [ ]:
# Export trained checkpoint
# !python export.py --google_drive --drive_root {DRIVE_ROOT}
